In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_care_plans;
SET search_path TO qdp_care_plans;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp_care_plans',
  '0.1',
  'care_plans.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_care_plan table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_care_plan CASCADE;

CREATE TABLE hub_care_plan (
    care_plan_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    care_plan_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_care_plan ADD CONSTRAINT pk_careplan_hubcareplan_careplanid PRIMARY KEY (care_plan_id);

COMMENT ON COLUMN hub_care_plan.care_plan_id IS 'Care Plan Identity';




-- -----------------------------------------------------------
-- create sat_care_plan table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan;

CREATE TABLE sat_care_plan (
    sat_care_plan_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    care_plan_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	encounter_id BIGINT,
	party_id BIGINT,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	intent_code VARCHAR,
	intent_concept_id BIGINT,
	intent_raw_code VARCHAR,
	intent_raw_concept_id BIGINT,
	title VARCHAR,
	description TEXT,
	created TIMESTAMP,
	custodian_party_id BIGINT,
	record_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan ADD CONSTRAINT pk_careplan_satcareplan_satcareplanid PRIMARY KEY (sat_care_plan_id);
ALTER TABLE sat_care_plan ADD CONSTRAINT fk_careplan_satcareplan_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan.sat_care_plan_id IS 'Care Plan Identity Sat';
COMMENT ON COLUMN sat_care_plan.encounter_id IS 'The Encounter during which this CarePlan was created or to which the creation of this record is tightly associated. See https://build.fhir.org/careplan-definitions.html#CarePlan.encounter';
COMMENT ON COLUMN sat_care_plan.party_id IS 'The Patient Identifier. Who the care plan is for. See https://build.fhir.org/careplan-definitions.html#CarePlan.subject';
COMMENT ON COLUMN sat_care_plan.status_code IS 'Status of the Care Plan. E.g. draft, active, on-hold, revoked, completed, entered-in-error, unknown. See https://hl7.org/fhir/R5/valueset-request-status.html';
COMMENT ON COLUMN sat_care_plan.intent_code IS 'Level of authority/intentionality associated with the care plan. E.g. proposal, plan, order, option, directive. See https://hl7.org/fhir/R5/valueset-care-plan-intent.html';
COMMENT ON COLUMN sat_care_plan.title IS 'Human-friendly name for the care plan. See https://build.fhir.org/careplan-definitions.html#CarePlan.title';
COMMENT ON COLUMN sat_care_plan.description IS 'Summary of nature of plan. See https://build.fhir.org/careplan-definitions.html#CarePlan.description';
COMMENT ON COLUMN sat_care_plan.created IS 'Date record was first recorded. See https://build.fhir.org/careplan-definitions.html#CarePlan.created';
COMMENT ON COLUMN sat_care_plan.custodian_party_id IS 'Who is the designated responsible party. E.g. Patient, Practitioner, PractitionerRole, Device, Organization, CareTeam. See https://build.fhir.org/careplan-definitions.html#CarePlan.custodian';
COMMENT ON COLUMN sat_care_plan.record_id IS 'A Unique Identifier for a Data Record within the System that allows Data Ownership, Ownership History, Dynamic Attributes and Metadata to be associated with that Record';



-- -----------------------------------------------------------
-- create sat_care_plan_category table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_category;

CREATE TABLE sat_care_plan_category (
	care_plan_category_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	category_code VARCHAR,
	category_concept_id BIGINT,
	category_raw_code VARCHAR,
	category_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_category ADD CONSTRAINT pk_careplan_satcareplancategory_careplancategoryid PRIMARY KEY (care_plan_category_id);
ALTER TABLE sat_care_plan_category ADD CONSTRAINT fk_careplan_satcareplancategory_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_category.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_category.care_plan_category_id IS 'Care Plan Category Identity';
COMMENT ON COLUMN sat_care_plan_category.category_code IS 'Type of plan. E.g. assess-plan (Assessment and Plan of Treatment), handp (History and Physical), discharge-summary, progress-note, consult-note. See https://hl7.org/fhir/R5/valueset-care-plan-category.html';
COMMENT ON COLUMN sat_care_plan_category.rank IS 'The recommended order the Care Plan Category Codes should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_addresses table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_addresses;

CREATE TABLE sat_care_plan_addresses (
	care_plan_addresses_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	condition_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_addresses ADD CONSTRAINT pk_careplan_satcareplanaddresses_careplanaddressesid PRIMARY KEY (care_plan_addresses_id);
ALTER TABLE sat_care_plan_addresses ADD CONSTRAINT fk_careplan_satcareplanaddresses_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_addresses.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_addresses.care_plan_addresses_id IS 'Care Plan Addresses Identity';
COMMENT ON COLUMN sat_care_plan_addresses.condition_id IS 'Health issues this plan addresses. Identifies the conditions/problems/concerns/diagnoses/etc. whose management and/or mitigation are handled by this plan. Cross-schema reference to Condition. See https://build.fhir.org/careplan-definitions.html#CarePlan.addresses';
COMMENT ON COLUMN sat_care_plan_addresses.rank IS 'The recommended order the Care Plan Addresses entries should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_activity table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_activity;

CREATE TABLE sat_care_plan_activity (
	care_plan_activity_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	performed_activity_code VARCHAR,
	performed_activity_concept_id BIGINT,
	performed_activity_raw_code VARCHAR,
	performed_activity_raw_concept_id BIGINT,
	planned_activity_reference_type VARCHAR,
	planned_activity_reference_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_activity ADD CONSTRAINT pk_careplan_satcareplanactivity_careplanactivityid PRIMARY KEY (care_plan_activity_id);
ALTER TABLE sat_care_plan_activity ADD CONSTRAINT fk_careplan_satcareplanactivity_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_activity.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_activity.care_plan_activity_id IS 'Care Plan Activity Identity';
COMMENT ON COLUMN sat_care_plan_activity.performed_activity_code IS 'Results of the activity (Observation, DiagnosticReport, etc.) or a reference to the resource representing the activity performed. See https://build.fhir.org/careplan-definitions.html#CarePlan.activity.performedActivity';
COMMENT ON COLUMN sat_care_plan_activity.planned_activity_reference_type IS 'Type of the planned activity resource. E.g. Appointment, CommunicationRequest, DeviceRequest, MedicationRequest, NutritionOrder, Task, ServiceRequest, VisionPrescription, ImmunizationRecommendation, RequestOrchestration. See https://build.fhir.org/careplan-definitions.html#CarePlan.activity.plannedActivityReference';
COMMENT ON COLUMN sat_care_plan_activity.planned_activity_reference_id IS 'Identifier of the planned activity resource';
COMMENT ON COLUMN sat_care_plan_activity.rank IS 'The recommended order the Care Plan Activity entries should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_care_team table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_care_team;

CREATE TABLE sat_care_plan_care_team (
	care_plan_care_team_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	care_team_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_care_team ADD CONSTRAINT pk_careplan_satcareplancareteam_careplancareteamid PRIMARY KEY (care_plan_care_team_id);
ALTER TABLE sat_care_plan_care_team ADD CONSTRAINT fk_careplan_satcareplancareteam_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_care_team.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_care_team.care_plan_care_team_id IS 'Care Plan Care Team Identity';
COMMENT ON COLUMN sat_care_plan_care_team.care_team_id IS 'Identifies all people and organizations who are expected to be involved in the care envisioned by this plan. Cross-schema reference to CareTeam. See https://build.fhir.org/careplan-definitions.html#CarePlan.careTeam';
COMMENT ON COLUMN sat_care_plan_care_team.rank IS 'The recommended order the Care Plan Care Team entries should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_contributor table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_contributor;

CREATE TABLE sat_care_plan_contributor (
	care_plan_contributor_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	contributor_party_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_contributor ADD CONSTRAINT pk_careplan_satcareplancontributor_careplancontributorid PRIMARY KEY (care_plan_contributor_id);
ALTER TABLE sat_care_plan_contributor ADD CONSTRAINT fk_careplan_satcareplancontributor_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_contributor.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_contributor.care_plan_contributor_id IS 'Care Plan Contributor Identity';
COMMENT ON COLUMN sat_care_plan_contributor.contributor_party_id IS 'Who provided the content of the care plan. E.g. Patient, Practitioner, PractitionerRole, Device, RelatedPerson, Organization, CareTeam. See https://build.fhir.org/careplan-definitions.html#CarePlan.contributor';
COMMENT ON COLUMN sat_care_plan_contributor.rank IS 'The recommended order the Care Plan Contributor entries should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_goal table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_goal;

CREATE TABLE sat_care_plan_goal (
	care_plan_goal_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	goal_description_code VARCHAR,
	goal_description_concept_id BIGINT,
	goal_description_raw_code VARCHAR,
	goal_description_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_goal ADD CONSTRAINT pk_careplan_satcareplangoal_careplangoalid PRIMARY KEY (care_plan_goal_id);
ALTER TABLE sat_care_plan_goal ADD CONSTRAINT fk_careplan_satcareplangoal_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_goal.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_goal.care_plan_goal_id IS 'Care Plan Goal Identity';
COMMENT ON COLUMN sat_care_plan_goal.goal_description_code IS 'Desired outcome of plan. Describes the intended objective(s) of carrying out the care plan (SNOMED CT clinical findings coded). See https://build.fhir.org/careplan-definitions.html#CarePlan.goal';
COMMENT ON COLUMN sat_care_plan_goal.rank IS 'The recommended order the Care Plan Goal entries should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_supporting_info table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_supporting_info;

CREATE TABLE sat_care_plan_supporting_info (
	care_plan_supporting_info_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	identifier_type VARCHAR,
	identifier BIGINT,
	reference VARCHAR,
	notes TEXT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_supporting_info ADD CONSTRAINT pk_careplan_satcareplansuppinfo_careplansuppinfoid PRIMARY KEY (care_plan_supporting_info_id);
ALTER TABLE sat_care_plan_supporting_info ADD CONSTRAINT fk_careplan_satcareplansuppinfo_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_supporting_info.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_supporting_info.care_plan_supporting_info_id IS 'Care Plan Supporting Info Identity';
COMMENT ON COLUMN sat_care_plan_supporting_info.identifier_type IS 'Type the reference refers to (e.g. "Patient") - must be a resource in resources. See https://hl7.org/fhir/R5/valueset-resource-types.html';
COMMENT ON COLUMN sat_care_plan_supporting_info.identifier IS 'Logical reference, when literal reference is not known';
COMMENT ON COLUMN sat_care_plan_supporting_info.reference IS 'Literal reference, Relative, internal or absolute URL';
COMMENT ON COLUMN sat_care_plan_supporting_info.notes IS 'Human Actor entered supporting information about the care plan. See https://build.fhir.org/careplan-definitions.html#CarePlan.supportingInfo';
COMMENT ON COLUMN sat_care_plan_supporting_info.rank IS 'The recommended order the Care Plan Supporting Info entries should be presented';



-- -----------------------------------------------------------
-- create sat_care_plan_notes table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_care_plan_notes;

CREATE TABLE sat_care_plan_notes (
	care_plan_note_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	care_plan_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	author_reference_party_id BIGINT,
	author_string VARCHAR,
	time TIMESTAMP,
	text TEXT NOT NULL,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_care_plan_notes ADD CONSTRAINT pk_careplan_satcareplannotes_careplannoteid PRIMARY KEY (care_plan_note_id);
ALTER TABLE sat_care_plan_notes ADD CONSTRAINT fk_careplan_satcareplannotes_hubcareplan_careplanid FOREIGN KEY (care_plan_id) REFERENCES hub_care_plan(care_plan_id);

COMMENT ON COLUMN sat_care_plan_notes.care_plan_id IS 'Care Plan Identity';
COMMENT ON COLUMN sat_care_plan_notes.care_plan_note_id IS 'Care Plan Note Identity';
COMMENT ON COLUMN sat_care_plan_notes.author_reference_party_id IS 'The clinical Party Identifier that recorded this information. E.g. Practitioner, PractitionerRole, Patient, RelatedPerson, Organization. See https://build.fhir.org/careplan-definitions.html#CarePlan.note';
COMMENT ON COLUMN sat_care_plan_notes.author_string IS 'The Individual that recorded this information (as free text)';
COMMENT ON COLUMN sat_care_plan_notes.time IS 'When this information was recorded';
COMMENT ON COLUMN sat_care_plan_notes.text IS 'General notes about the care plan not captured elsewhere. See https://build.fhir.org/careplan-definitions.html#CarePlan.note';




-- -----------------------------------------------------------
-- create view_care_plans view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_plans AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_plan_id,
	h.care_plan_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_care_plan_id,
  s.encounter_id,
  s.party_id,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.intent_code,
  s.intent_concept_id,
  s.intent_raw_code,
  s.intent_raw_concept_id,
  s.title,
  s.description,
  s.created,
  s.custodian_party_id,
  s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_plans.hub_care_plan h
    JOIN budd.qdp_care_plans.sat_care_plan s
      ON h.care_plan_id = s.care_plan_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_plans;


-- -----------------------------------------------------------
-- create view_care_plan_categories view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_plan_categories AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_plan_id,
	h.care_plan_entity_id,
  s.status_code,
  s.intent_code,
  s.title,
	cc.care_plan_category_id,
	cc.category_code,
	cc.category_concept_id,
	cc.category_raw_code,
	cc.category_raw_concept_id,
	cc.rank AS category_rank,
	cc.period_start AS category_period_start,
	cc.period_end AS category_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_plans.hub_care_plan h
    JOIN budd.qdp_care_plans.sat_care_plan s
      ON h.care_plan_id = s.care_plan_id
    JOIN budd.qdp_care_plans.sat_care_plan_category cc
      ON h.care_plan_id = cc.care_plan_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_plan_categories;


-- -----------------------------------------------------------
-- create view_care_plan_activities view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_plan_activities AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_plan_id,
	h.care_plan_entity_id,
  s.status_code,
  s.intent_code,
  s.title,
	ca.care_plan_activity_id,
	ca.performed_activity_code,
	ca.performed_activity_concept_id,
	ca.performed_activity_raw_code,
	ca.performed_activity_raw_concept_id,
	ca.planned_activity_reference_type,
	ca.planned_activity_reference_id,
	ca.rank AS activity_rank,
	ca.period_start AS activity_period_start,
	ca.period_end AS activity_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_plans.hub_care_plan h
    JOIN budd.qdp_care_plans.sat_care_plan s
      ON h.care_plan_id = s.care_plan_id
    JOIN budd.qdp_care_plans.sat_care_plan_activity ca
      ON h.care_plan_id = ca.care_plan_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_plan_activities;


-- -----------------------------------------------------------
-- create view_care_plan_goals view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_care_plan_goals AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.care_plan_id,
	h.care_plan_entity_id,
  s.status_code,
  s.intent_code,
  s.title,
	cg.care_plan_goal_id,
	cg.goal_description_code,
	cg.goal_description_concept_id,
	cg.goal_description_raw_code,
	cg.goal_description_raw_concept_id,
	cg.rank AS goal_rank,
	cg.period_start AS goal_period_start,
	cg.period_end AS goal_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_care_plans.hub_care_plan h
    JOIN budd.qdp_care_plans.sat_care_plan s
      ON h.care_plan_id = s.care_plan_id
    JOIN budd.qdp_care_plans.sat_care_plan_goal cg
      ON h.care_plan_id = cg.care_plan_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_care_plan_goals;


/**

**/
